vector open order combine

In [60]:
import pandas as pd 
import os
import json

In [61]:
current_dir = os.getcwd()
#current_dir = "/home/VECVNET/zzasom/jupyter/01_Development_work/Black_Forecasting/Deployment/Build/"
cred_path = os.path.join(current_dir, "OO_manual.json")
#cred_path = "/home/VECVNET/zzasom/jupyter/01_Development_work/Black_Forecasting/Deployment/Build/cred.json"

# Load the configuration
with open(cred_path, "r") as config_file:
    config = json.load(config_file)

VF_rationed_path = config.get("VF_rationed", "")
VF_openso_path = config.get("VF_openso", "")
credit_block_path = config.get("credit_block", "")
zfmo_path = config.get("zfmo", "")


In [64]:
# Read the actual Excel files into pandas DataFrames.
open_order_1 = pd.read_csv(VF_rationed_path)
open_order_2 = pd.read_csv(VF_openso_path)
credit = pd.read_excel(credit_block_path)
zfmo_mapping_df = pd.read_excel(zfmo_path)

print("VF rationed shape:", open_order_1.shape)
print("VF open SO shape:", open_order_2.shape)
print("Credit block shape:", credit.shape)
print("ZFMO mapping shape:", zfmo_mapping_df.shape)

VF rationed shape: (36363, 8)
VF open SO shape: (8673, 10)
Credit block shape: (216, 11)
ZFMO mapping shape: (530655, 4)


In [65]:
import os
import sys
from pathlib import Path

build_dir = Path.cwd()
src_dir = build_dir / "src"

os.environ["PYTHONPATH"] = str(src_dir)

print("Python executable:", sys.executable)
print("PYTHONPATH:", os.environ["PYTHONPATH"])

!{sys.executable} scripts/run_data_cleaning.py --config config/data_cleaning.yaml --only zfmo_dealer_material

Python executable: /home/VECVNET/zzasom/anaconda3/bin/python
PYTHONPATH: /home/VECVNET/zzasom/jupyter/01_Developement_work_QA/Chakshu_G/order_orchestration/order_orchestration_mannual/Build/src
ZFMO mapping input rows: 530655
ZFMO mapping invalid rows removed: 0
ZFMO mapping duplicate rows removed: 0
ZFMO mapping unique dealer-material combinations: 530655

Data cleaning completed successfully.
Generated outputs:
  - zfmo_dealer_material: 530655 rows x 2 columns


In [66]:
zfmo_mapping = pd.read_excel("/home/VECVNET/zzasom/jupyter/01_Developement_work_QA/Chakshu_G/order_orchestration/order_orchestration_mannual/Build/data/cleaned/cleaned_zfmo_dealer_material.xlsx")

In [67]:
print("ZFMO columns:", zfmo_mapping.columns.tolist())

ZFMO columns: ['dealer_code', 'material_code']


In [3]:
# open_order_2 = pd.read_csv(r"C:\Order Orchestration code\order_orchestration\Regular_original_files\13_11_2026\ECC_OPENSO_20261113.CSV")

In [4]:
# credit = pd.read_excel(r"C:\Order Orchestration code\order_orchestration\Regular_original_files\13_11_2026\credit block 13112026.xlsx",engine='openpyxl')

In [5]:
# zfmo_mapping = pd.read_excel(r"C:\Order Orchestration code\order_orchestration\data\cleaned\cleaned_zfmo_dealer_material.xlsx",engine='openpyxl')

In [68]:
import pandas as pd

rationed_qty_mapped_df = open_order_1.copy()
zfmo_mapping_df = zfmo_mapping.copy()


def normalize_code(series: pd.Series) -> pd.Series:
    return (
        series
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\s+", "", regex=True)
    )


# Normalize rationed quantity matching fields.
rationed_qty_mapped_df["To Location"] = normalize_code(
    rationed_qty_mapped_df["To Location"]
)

rationed_qty_mapped_df["Material Code"] = normalize_code(
    rationed_qty_mapped_df["Material Code"]
)

rationed_qty_mapped_df["Order Type"] = normalize_code(
    rationed_qty_mapped_df["Order Type"]
)

# Normalize ZFMO master matching fields.
zfmo_mapping_df["dealer_code"] = normalize_code(
    zfmo_mapping_df["dealer_code"]
)

zfmo_mapping_df["material_code"] = normalize_code(
    zfmo_mapping_df["material_code"]
)

# Create unique ZFMO dealer-material keys.
zfmo_keys = pd.MultiIndex.from_frame(
    zfmo_mapping_df[
        ["dealer_code", "material_code"]
    ].drop_duplicates()
)

# Find dealer-material matches.
zfmo_match_mask = pd.MultiIndex.from_frame(
    rationed_qty_mapped_df[
        ["To Location", "Material Code"]
    ]
).isin(zfmo_keys)

# Store the final override mask BEFORE changing Order Type.
zfmo_override_mask = (
    zfmo_match_mask
    & rationed_qty_mapped_df["Order Type"].eq("ZMSO")
)

print(
    "Dealer-material matches found:",
    int(zfmo_match_mask.sum()),
)

print(
    "ZMSO rows to change to ZFMO:",
    int(zfmo_override_mask.sum()),
)

# Apply the override.
rationed_qty_mapped_df.loc[
    zfmo_override_mask,
    "Order Type",
] = "ZFMO"

open_order_1 = rationed_qty_mapped_df.copy()

print(
    "Rows changed from ZMSO to ZFMO:",
    int(zfmo_override_mask.sum()),
)

Dealer-material matches found: 10679
ZMSO rows to change to ZFMO: 10679
Rows changed from ZMSO to ZFMO: 10679


In [69]:
# rationed_qty_mapped_df.to_csv(
#     r"C:\Order Orchestration code\order_orchestration\data\cleaned\temp_rationed_qty_mapped.csv",
#     index=False,
# )

In [70]:
open_order_1.head(5)

,From Location,Material Code,To Location,Quantity,Colour Code,Order Type,DATE,TIME
0,1158,IA203813,1S1127,1,3,ZFMO,20261113,30209
1,1153,IA203813,1S4046,1,4,ZFMO,20261113,30209
2,1153,IA203813,1S4048,1,3,ZFMO,20261113,30209
3,1156,IA204306,1S4098,1,9,ZMSO,20261113,30209
4,1156,IA204306,1S4098,1,2,ZMSO,20261113,30209


In [71]:
open_order_1.columns

Index(['From Location', 'Material Code', 'To Location', 'Quantity',
       'Colour Code', 'Order Type', 'DATE', 'TIME'],
      dtype='object')

In [72]:
open_order_1.rename(columns = {'From Location':'Plnt','Material Code':"Material   ","To Location":"Customer","Quantity":"Open Order Qty.","Colour Code":"EAN/UPC","Order Type":"SaTy","DATE":"Date      ","TIME":"Time    "}, inplace = True)

In [73]:
open_order_1.columns

Index(['Plnt', 'Material   ', 'Customer', 'Open Order Qty.', 'EAN/UPC', 'SaTy',
       'Date      ', 'Time    '],
      dtype='object')

In [74]:
open_order_2.columns

Index(['Customer', 'Order type', 'Order No.', 'Order Item', 'Create Date',
       'Creating Time.', 'Material', 'Order Qty.', 'Plant', 'UOM'],
      dtype='object')

In [75]:
open_order_2.rename(columns = {'Customer':'Customer','Order type':'SaTy','Create Date':'Doc. Date ','Creating Time.': 'Time    .1','Material':'Material   ','Order Qty.':'Open Order Qty.','Plant':'Plnt'}, inplace = True)

In [76]:
open_order_2.columns

Index(['Customer', 'SaTy', 'Order No.', 'Order Item', 'Doc. Date ',
       'Time    .1', 'Material   ', 'Open Order Qty.', 'Plnt', 'UOM'],
      dtype='object')

In [77]:
open_order_2 = open_order_2[['Customer','SaTy','Doc. Date ','Time    .1','Material   ','Open Order Qty.','Plnt']]

In [78]:
open_order_1.columns

Index(['Plnt', 'Material   ', 'Customer', 'Open Order Qty.', 'EAN/UPC', 'SaTy',
       'Date      ', 'Time    '],
      dtype='object')

In [79]:
open_order_2.columns

Index(['Customer', 'SaTy', 'Doc. Date ', 'Time    .1', 'Material   ',
       'Open Order Qty.', 'Plnt'],
      dtype='object')

In [80]:
# Date
open_order_1["Date      "] = pd.to_datetime(
    open_order_1["Date      "].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

# Time
open_order_1["Time    "] = (
    open_order_1["Time    "]
    .astype(str)
    .str.zfill(6)
    .str.replace(r"(\d{2})(\d{2})(\d{2})", r"\1:\2:\3", regex=True)
)

# EAN/UPC: 1-10 -> 11-20
mask = open_order_1["EAN/UPC"].between(1, 10)
open_order_1.loc[mask, "EAN/UPC"] = open_order_1.loc[mask, "EAN/UPC"] + 10

In [81]:
open_order_1['Doc. Date '] = open_order_1['Date      ']
open_order_1['Time    .1'] = open_order_1['Time    ']

In [82]:
open_order_1.columns

Index(['Plnt', 'Material   ', 'Customer', 'Open Order Qty.', 'EAN/UPC', 'SaTy',
       'Date      ', 'Time    ', 'Doc. Date ', 'Time    .1'],
      dtype='object')

In [83]:
open_order_2.columns

Index(['Customer', 'SaTy', 'Doc. Date ', 'Time    .1', 'Material   ',
       'Open Order Qty.', 'Plnt'],
      dtype='object')

In [84]:
open_order_2.head()

,Customer,SaTy,Doc. Date,Time .1,Material,Open Order Qty.,Plnt
0,1XS1299,ZESS,20241114,115231,IA236530,1.0,1110
1,1S6265,ZEOR,20260819,101116,ST000213M,1.0,1110
2,1S6228,ZMSO,20260818,184213,ID379455,2.0,1110
3,1S2150,ZROH,20260811,171820,ST111341,1.0,1110
4,1S3452,ZROH,20260813,124119,IA460255,1.0,1110


In [85]:
# Date
open_order_2["Doc. Date "] = pd.to_datetime(
    open_order_2["Doc. Date "].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

# Time
open_order_2["Time    .1"] = (
    open_order_2["Time    .1"]
    .astype(str)
    .str.zfill(6)
    .str.replace(r"(\d{2})(\d{2})(\d{2})", r"\1:\2:\3", regex=True)
)

In [86]:
open_order_2['EAN/UPC'] = 1
open_order_2['Date      '] = open_order_1['Date      ']
open_order_2['Time    '] = open_order_1['Time    ']

In [87]:
open_order_2.head(5)

,Customer,SaTy,Doc. Date,Time .1,Material,Open Order Qty.,Plnt,EAN/UPC,Date,Time
0,1XS1299,ZESS,2024-11-14,11:52:31,IA236530,1.0,1110,1,2026-11-13,03:02:09
1,1S6265,ZEOR,2026-08-19,10:11:16,ST000213M,1.0,1110,1,2026-11-13,03:02:09
2,1S6228,ZMSO,2026-08-18,18:42:13,ID379455,2.0,1110,1,2026-11-13,03:02:09
3,1S2150,ZROH,2026-08-11,17:18:20,ST111341,1.0,1110,1,2026-11-13,03:02:09
4,1S3452,ZROH,2026-08-13,12:41:19,IA460255,1.0,1110,1,2026-11-13,03:02:09


In [88]:
open_order = pd.concat([open_order_2, open_order_1], ignore_index=True)

In [89]:
open_order['Cust Type']='DOMESTIC'
open_order['Material Description                    ']='ABCD'
open_order['Name 1                        ']='XYZ'
open_order['CustomerStatus'] = 0

In [90]:
credit.head()

,Order Date,CP Code,CP Name,City,Order Type,Order Type Desc,Order No,Order Qty,Net Value,Net Value(MINR),Plant
0,13.11.2026,1S5110,Kesharwani Automotive Pvt Ltd,Shahdol,ZMSO,Normal,1212984134,10,22586.48,0.023,1110
1,13.11.2026,1S5080,Prayag Automobiles,Katni,ZMSO,Normal,1212984138,5,13397.44,0.013,1110
2,13.11.2026,1S3369,AJMERA AUTO SERVICES,KARAD,ZMSO,Normal,1212984139,4,19106.00,0.019,1110
3,13.11.2026,1S1191,JANDU MOTORS,Rajasthan,ZMSO,Normal,1212984140,1,333.06,0.000,1110
4,13.11.2026,1S7040,VANTAGE COMMERCIAL VEHICLES,BHILWARA,ZMSO,Normal,1212984141,25,54904.30,0.055,1110


In [91]:
credit.columns

Index(['Order Date', 'CP Code', 'CP Name                            ',
       'City                            ', 'Order Type', 'Order Type Desc',
       'Order No  ', ' Order Qty', '    Net Value', 'Net Value(MINR)',
       'Plant'],
      dtype='object')

In [92]:
credit = credit[['CP Code']]

In [93]:
credit.head()

,CP Code
0,1S5110
1,1S5080
2,1S3369
3,1S1191
4,1S7040


In [94]:
# Create set of credit block dealers
credit_block_customers = set(
    credit["CP Code"].astype(str).str.strip()
)

# Flag in open order file
open_order["CustomerStatus"] = (
    open_order["Customer"]
    .astype(str)
    .str.strip()
    .isin(credit_block_customers)
    .astype(int)
)

In [95]:
open_order.head(10)

,Customer,SaTy,Doc. Date,Time .1,Material,Open Order Qty.,Plnt,EAN/UPC,Date,Time,Cust Type,Material Description,Name 1,CustomerStatus
0,1XS1299,ZESS,2024-11-14,11:52:31,IA236530,1.0,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
1,1S6265,ZEOR,2026-08-19,10:11:16,ST000213M,1.0,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
2,1S6228,ZMSO,2026-08-18,18:42:13,ID379455,2.0,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
3,1S2150,ZROH,2026-08-11,17:18:20,ST111341,1.0,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
4,1S3452,ZROH,2026-08-13,12:41:19,IA460255,1.0,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
5,1S1251,ZROH,2026-08-14,11:16:21,ID315753,1.0,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,1
6,1XS1182,ZESS,2026-08-13,16:39:36,IU481982,10.0,9045,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
7,1XS1182,ZESS,2026-08-13,16:39:36,ID319332,1.0,9045,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
8,1XS1182,ZESS,2026-08-13,16:39:36,IA222133,1.0,9045,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
9,1XS1182,ZESS,2026-08-13,16:39:36,ID313220,1.0,9045,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0


In [96]:
open_order.dtypes

Customer                                            object
SaTy                                                object
Doc. Date                                   datetime64[ns]
Time    .1                                          object
Material                                            object
Open Order Qty.                                    float64
Plnt                                                object
EAN/UPC                                              int64
Date                                        datetime64[ns]
Time                                                object
Cust Type                                           object
Material Description                                object
Name 1                                              object
CustomerStatus                                       int64
dtype: object

In [97]:
open_order.to_excel(r"/home/VECVNET/zzasom/jupyter/01_Developement_work_QA/Chakshu_G/order_orchestration/order_orchestration_mannual/Input/modified_input_files/input_open_order_mapped_ipynb.xlsx", index=False)